# NovaMarket — Análisis de Datos con Python

## Objetivo

Explorar, analizar y visualizar los datos de ventas de NovaMarket utilizando Python, a partir de la información previamente limpiada y validada en PostgreSQL.

El análisis integra consultas a PostgreSQL, procesamiento con Pandas, análisis exploratorio y generación de indicadores para su posterior visualización en Power BI.

## 1. Conexión y carga de datos

Se establece una conexión segura con PostgreSQL y se carga la tabla `fact_ventas_analitica`, que contiene los registros después del proceso de limpieza.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from urllib.parse import quote_plus
from getpass import getpass
from sqlalchemy import create_engine

In [ ]:
usuario = "postgres"
host = "localhost"
puerto = "5432"
base_datos = "novamarket"

password = getpass("Introduce la contraseña de PostgreSQL: ")

engine = create_engine(
    f"postgresql+psycopg2://{usuario}:{quote_plus(password)}@{host}:{puerto}/{base_datos}"
)

print("Conexión preparada correctamente.")

In [ ]:
df = pd.read_sql_query(
    """
    SELECT *
    FROM fact_ventas_analitica;
    """,
    engine
)

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Preparación de métricas comerciales

Se calculan por separado la venta bruta, el monto del descuento y la venta neta. Para el análisis comercial se utiliza `venta` como sinónimo de venta neta, de forma que los resultados de Python sean consistentes con los utilizados posteriormente en Power BI.

In [ ]:
df["venta_bruta"] = (
    df["cantidad"] * df["precio_final_unitario"]
)

df["descuento_monto"] = (
    df["venta_bruta"] * df["descuento"]
)

df["venta_neta"] = (
    df["venta_bruta"] - df["descuento_monto"]
)

df["venta"] = df["venta_neta"]

df[[
    "cantidad",
    "precio_final_unitario",
    "descuento",
    "venta_bruta",
    "descuento_monto",
    "venta_neta"
]].head()

In [ ]:
ventas_totales = df["venta"].sum()
unidades_vendidas = df["cantidad"].sum()
ordenes_totales = df["orden_id"].nunique()
clientes_unicos = df["cliente_id"].nunique()
productos_unicos = df["producto_id"].nunique()
ticket_promedio = ventas_totales / ordenes_totales

print("Ventas netas:", round(ventas_totales, 2))
print("Unidades vendidas:", unidades_vendidas)
print("Órdenes únicas:", ordenes_totales)
print("Clientes únicos:", clientes_unicos)
print("Productos únicos:", productos_unicos)
print("Ticket promedio:", round(ticket_promedio, 2))

## 3. Análisis temporal

In [ ]:
ventas_mensuales = (
    df.groupby(df["fecha_compra"].dt.to_period("M"))["venta"]
      .sum()
)

ventas_mensuales

In [ ]:
ventas_mensuales.plot(
    kind="line",
    figsize=(12, 5),
    title="Evolución mensual de ventas netas"
)

plt.xlabel("Mes")
plt.ylabel("Ventas netas")
plt.tight_layout()
plt.show()

In [ ]:
mejor_mes = ventas_mensuales.idxmax()
peor_mes = ventas_mensuales.idxmin()

print("Mejor mes:", mejor_mes, "con ventas de $", round(ventas_mensuales.max(), 2))
print("Peor mes:", peor_mes, "con ventas de $", round(ventas_mensuales.min(), 2))

In [ ]:
caida_mejor_peor = (
    (ventas_mensuales.min() - ventas_mensuales.max())
    / ventas_mensuales.max()
) * 100

print(
    "Variación entre el mejor y peor mes:",
    round(caida_mejor_peor, 2),
    "%"
)

## 4. Análisis por método de pago

In [ ]:
ventas_metodo_pago = (
    df.groupby("metodo_pago")["venta"]
      .sum()
      .sort_values(ascending=False)
)

porcentaje_metodo_pago = (
    ventas_metodo_pago / ventas_metodo_pago.sum() * 100
).round(2)

ventas_metodo_pago

In [ ]:
ventas_metodo_pago.plot(
    kind="bar",
    figsize=(10, 5),
    title="Ventas netas por método de pago"
)

plt.xlabel("Método de pago")
plt.ylabel("Ventas netas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
ordenes_metodo_pago = (
    df.groupby("metodo_pago")["orden_id"]
      .nunique()
      .sort_values(ascending=False)
)

ticket_metodo_pago = (
    ventas_metodo_pago / ordenes_metodo_pago
).round(2)

ticket_metodo_pago

In [ ]:
ticket_metodo_pago.plot(
    kind="bar",
    figsize=(10, 5),
    title="Ticket promedio por método de pago"
)

plt.xlabel("Método de pago")
plt.ylabel("Ticket promedio")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Análisis de clientes

In [ ]:
ordenes_por_cliente = (
    df.groupby("cliente_id")["orden_id"]
      .nunique()
)

ventas_por_cliente = (
    df.groupby("cliente_id")["venta"]
      .sum()
      .sort_values(ascending=False)
)

print("Clientes únicos:", ventas_por_cliente.size)
print("Órdenes promedio por cliente:", round(ordenes_por_cliente.mean(), 2))

ventas_por_cliente.head(10)

In [ ]:
ordenes_por_cliente.describe()

In [ ]:
top_10_clientes = ventas_por_cliente.head(10)

participacion_top_10 = (
    top_10_clientes.sum() / ventas_por_cliente.sum() * 100
)

print("Ventas de los 10 clientes principales:", round(top_10_clientes.sum(), 2))
print("Participación en ventas totales:", round(participacion_top_10, 2), "%")

In [ ]:
cliente_top = ventas_por_cliente.idxmax()
venta_cliente_top = ventas_por_cliente.max()
cliente_top_ordenes = df[df["cliente_id"] == cliente_top]["orden_id"].nunique()
cliente_top_ticket = venta_cliente_top / cliente_top_ordenes
participacion_cliente_top = venta_cliente_top / ventas_por_cliente.sum() * 100

print("Cliente con mayor facturación:", cliente_top)
print("Facturación:", round(venta_cliente_top, 2))
print("Órdenes del cliente top:", cliente_top_ordenes)
print("Ticket promedio:", round(cliente_top_ticket, 2))
print("Participación del cliente top:", round(participacion_cliente_top, 2), "%")

## 6. Análisis de productos y categorías

In [ ]:
ventas_por_producto = (
    df.groupby("producto_id")
      .agg(
          ventas=("venta", "sum"),
          unidades=("cantidad", "sum"),
          ordenes=("orden_id", "nunique")
      )
      .sort_values("ventas", ascending=False)
)

ventas_por_producto.head(10)

In [ ]:
productos_por_unidades = (
    df.groupby("producto_id")["cantidad"]
      .sum()
      .sort_values(ascending=False)
)

productos_por_unidades.head(10)

In [ ]:
precio_promedio_producto = (
    ventas_por_producto["ventas"] / ventas_por_producto["unidades"]
).sort_values(ascending=False)

precio_promedio_producto.head(10)

In [ ]:
top_facturacion = set(ventas_por_producto.head(10).index)
top_unidades = set(productos_por_unidades.head(10).index)

productos_en_ambos = top_facturacion.intersection(top_unidades)

print("Productos presentes en ambos Top 10:", len(productos_en_ambos))
print("IDs:", sorted(productos_en_ambos))

comparacion_productos = ventas_por_producto.loc[
    sorted(productos_en_ambos)
].copy()

comparacion_productos["venta_por_orden"] = (
    comparacion_productos["ventas"] /
    comparacion_productos["ordenes"]
).round(2)

comparacion_productos

In [ ]:
productos = pd.read_sql(
    """
    SELECT *
    FROM dim_productos;
    """,
    engine
)

print("Productos cargados:", productos.shape[0])

In [ ]:
ventas_por_categoria = (
    df.merge(
        productos[["producto_id", "categoria"]],
        on="producto_id",
        how="left"
    )
    .groupby("categoria")["venta"]
    .sum()
    .sort_values(ascending=False)
)

unidades_por_categoria = (
    df.merge(
        productos[["producto_id", "categoria"]],
        on="producto_id",
        how="left"
    )
    .groupby("categoria")["cantidad"]
    .sum()
    .sort_values(ascending=False)
)

porcentaje_categoria = (
    ventas_por_categoria / ventas_por_categoria.sum() * 100
).round(2)

ventas_por_categoria

## 7. Rentabilidad

In [ ]:
df_rentabilidad = df.merge(
    productos[["producto_id", "categoria", "costo"]],
    on="producto_id",
    how="left"
)

df_rentabilidad["costo_total"] = (
    df_rentabilidad["cantidad"] *
    df_rentabilidad["costo"]
)

df_rentabilidad["margen"] = (
    df_rentabilidad["venta"] -
    df_rentabilidad["costo_total"]
)

df_rentabilidad.head()

In [ ]:
rentabilidad_categoria = (
    df_rentabilidad
    .groupby("categoria")
    .agg(
        ventas=("venta", "sum"),
        costo=("costo_total", "sum"),
        margen=("margen", "sum")
    )
)

rentabilidad_categoria["margen_pct"] = (
    rentabilidad_categoria["margen"] /
    rentabilidad_categoria["ventas"] * 100
).round(2)

rentabilidad_categoria = rentabilidad_categoria.sort_values(
    "margen",
    ascending=False
)

rentabilidad_categoria

In [ ]:
margen_total = df_rentabilidad["margen"].sum()
costo_total = df_rentabilidad["costo_total"].sum()
margen_global_pct = margen_total / ventas_totales * 100

print("Ventas netas:", round(ventas_totales, 2))
print("Costo total:", round(costo_total, 2))
print("Margen total:", round(margen_total, 2))
print("Margen global:", round(margen_global_pct, 2), "%")

In [ ]:
rentabilidad_anual = (
    df_rentabilidad
    .assign(año=df_rentabilidad["fecha_compra"].dt.year)
    .groupby("año")
    .agg(
        ventas=("venta", "sum"),
        costo=("costo_total", "sum"),
        margen=("margen", "sum")
    )
)

rentabilidad_anual["margen_pct"] = (
    rentabilidad_anual["margen"] /
    rentabilidad_anual["ventas"] * 100
).round(2)

rentabilidad_anual

In [ ]:
margen_por_producto = (
    df_rentabilidad
    .groupby("producto_id")
    .agg(
        ventas=("venta", "sum"),
        costo=("costo_total", "sum"),
        margen=("margen", "sum"),
        unidades=("cantidad", "sum")
    )
    .sort_values("margen", ascending=False)
)

rentabilidad_producto = (
    margen_por_producto.reset_index()
    .merge(
        productos[["producto_id", "nombre_producto", "categoria"]],
        on="producto_id",
        how="left"
    )
)

rentabilidad_producto["margen_pct"] = (
    rentabilidad_producto["margen"] /
    rentabilidad_producto["ventas"] * 100
).round(2)

rentabilidad_producto.head(10)

In [ ]:
productos_margen_bajo = (
    rentabilidad_producto[
        rentabilidad_producto["ventas"] > rentabilidad_producto["ventas"].median()
    ]
    .sort_values("margen_pct", ascending=True)
    .head(10)
)

productos_margen_bajo[[
    "producto_id",
    "nombre_producto",
    "categoria",
    "ventas",
    "margen",
    "margen_pct"
]]

In [ ]:
productos_alto_rendimiento = (
    rentabilidad_producto[
        (rentabilidad_producto["ventas"] > rentabilidad_producto["ventas"].median()) &
        (rentabilidad_producto["margen_pct"] > rentabilidad_producto["margen_pct"].median())
    ]
    .sort_values(
        ["ventas", "margen_pct"],
        ascending=False
    )
)

productos_alto_rendimiento[[
    "producto_id",
    "nombre_producto",
    "categoria",
    "ventas",
    "margen",
    "margen_pct"
]].head(15)

## 8. Segmentación de clientes

In [ ]:
segmentacion_clientes = ventas_por_cliente.to_frame(name="ventas")

segmentacion_clientes["segmento"] = pd.qcut(
    segmentacion_clientes["ventas"],
    q=4,
    labels=["Bajo", "Medio-Bajo", "Medio-Alto", "Alto"]
)

segmentacion_clientes["ordenes"] = ordenes_por_cliente

segmentacion_clientes["score_ventas"] = pd.qcut(
    segmentacion_clientes["ventas"],
    q=4,
    labels=[1, 2, 3, 4]
).astype(int)

segmentacion_clientes["score_frecuencia"] = pd.qcut(
    segmentacion_clientes["ordenes"],
    q=4,
    labels=[1, 2, 3, 4],
    duplicates="drop"
).astype(int)

segmentacion_clientes["score_total"] = (
    segmentacion_clientes["score_ventas"] +
    segmentacion_clientes["score_frecuencia"]
)

segmentacion_clientes.head()

In [ ]:
cliente_analisis = segmentacion_clientes.copy()

cliente_analisis["segmento_valor"] = pd.cut(
    cliente_analisis["score_total"],
    bins=[1, 3, 5, 6, 8],
    labels=[
        "Bajo valor",
        "Potencial",
        "Frecuente",
        "Alto valor"
    ],
    include_lowest=True
)

resumen_valor_cliente = (
    cliente_analisis
    .groupby("segmento_valor", observed=False)
    .agg(
        clientes=("ventas", "count"),
        ventas_totales=("ventas", "sum"),
        venta_promedio=("ventas", "mean"),
        ordenes_promedio=("ordenes", "mean")
    )
)

resumen_valor_cliente["participacion_ventas"] = (
    resumen_valor_cliente["ventas_totales"] /
    resumen_valor_cliente["ventas_totales"].sum() * 100
).round(2)

resumen_valor_cliente

In [ ]:
top_clientes = (
    cliente_analisis
    .sort_values("ventas", ascending=False)
    .head(20)
    .copy()
)

top_clientes["ticket_promedio"] = (
    top_clientes["ventas"] /
    top_clientes["ordenes"]
).round(2)

top_clientes[[
    "ventas",
    "ordenes",
    "ticket_promedio",
    "segmento_valor"
]]

## 9. Evolución de órdenes y crecimiento mensual

In [ ]:
ordenes_mensuales = (
    df.groupby(df["fecha_compra"].dt.to_period("M"))["orden_id"]
      .nunique()
)

ordenes_mensuales

In [ ]:
crecimiento_mensual = (
    ventas_mensuales.pct_change() * 100
).round(2)

crecimiento_mensual

In [ ]:
mejor_crecimiento = crecimiento_mensual.dropna().idxmax()
valor_mejor_crecimiento = crecimiento_mensual.dropna().max()

mayor_caida = crecimiento_mensual.dropna().idxmin()
valor_mayor_caida = crecimiento_mensual.dropna().min()

print("Mayor crecimiento:", mejor_crecimiento)
print("Variación:", round(valor_mejor_crecimiento, 2), "%")

print("Mayor caída:", mayor_caida)
print("Variación:", round(valor_mayor_caida, 2), "%")

In [ ]:
crecimiento_mensual.plot(
    kind="bar",
    figsize=(14, 5),
    title="Crecimiento mensual de ventas netas"
)

plt.axhline(0, linewidth=1)
plt.xlabel("Mes")
plt.ylabel("Variación (%)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 10. Resumen de categorías

In [ ]:
resumen_categorias = pd.DataFrame({
    "ventas": ventas_por_categoria,
    "unidades": unidades_por_categoria,
    "ventas_pct": (
        ventas_por_categoria / ventas_por_categoria.sum() * 100
    ).round(2),
    "unidades_pct": (
        unidades_por_categoria / unidades_por_categoria.sum() * 100
    ).round(2),
    "margen": rentabilidad_categoria["margen"],
    "margen_pct": rentabilidad_categoria["margen_pct"]
})

resumen_categorias["diferencia_ventas_unidades"] = (
    resumen_categorias["ventas_pct"] -
    resumen_categorias["unidades_pct"]
).round(2)

resumen_categorias.sort_values(
    "ventas",
    ascending=False
)

## 11. Preparación de datos para Power BI

In [ ]:
kpis = {
    "ventas_totales": df["venta"].sum(),
    "clientes_unicos": df["cliente_id"].nunique(),
    "productos_unicos": df["producto_id"].nunique(),
    "ordenes_unicas": df["orden_id"].nunique(),
    "ticket_promedio": df["venta"].sum() / df["orden_id"].nunique(),
    "unidades_vendidas": df["cantidad"].sum(),
    "margen_total": df_rentabilidad["margen"].sum(),
    "margen_global_pct": (
        df_rentabilidad["margen"].sum()
        / df_rentabilidad["venta"].sum()
        * 100
    )
}

kpis_df = pd.DataFrame(
    kpis.items(),
    columns=["KPI", "Valor"]
)

kpis_df

In [ ]:
clientes_powerbi = (
    segmentacion_clientes
    .reset_index()
)

clientes_powerbi.columns = [
    str(col).strip().lower()
    for col in clientes_powerbi.columns
]

clientes_powerbi.head()

In [ ]:
productos_powerbi = (
    rentabilidad_producto
    [
        [
            "producto_id",
            "nombre_producto",
            "categoria",
            "ventas",
            "costo",
            "margen",
            "margen_pct",
            "unidades"
        ]
    ]
    .sort_values("ventas", ascending=False)
)

productos_powerbi.head(10)

In [ ]:
ventas_powerbi = pd.read_sql(
    """
    SELECT
        orden_id,
        linea_orden_id,
        fecha_compra,
        cliente_id,
        producto_id,
        cantidad,
        descuento,
        precio_final_unitario,
        metodo_pago
    FROM fact_ventas_analitica;
    """,
    engine
)

ventas_powerbi["venta"] = (
    ventas_powerbi["cantidad"] *
    ventas_powerbi["precio_final_unitario"] *
    (1 - ventas_powerbi["descuento"])
)

print("Filas:", ventas_powerbi.shape[0])
print("Columnas:", ventas_powerbi.shape[1])

ventas_powerbi.head()

In [ ]:
print("clientes_powerbi:", clientes_powerbi.shape)
print("productos_powerbi:", productos_powerbi.shape)
print("ventas_powerbi:", ventas_powerbi.shape)

In [ ]:
clientes_powerbi.to_csv(
    "clientes_powerbi.csv",
    index=False,
    encoding="utf-8-sig"
)

productos_powerbi.to_csv(
    "productos_powerbi.csv",
    index=False,
    encoding="utf-8-sig"
)

ventas_powerbi.to_csv(
    "ventas_powerbi.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos exportados correctamente.")

## 12. Validación final

Se comprueba que el conjunto preparado para Power BI conserve el mismo número de filas y el mismo total de ventas netas que el DataFrame principal.

In [ ]:
print("Filas df:", len(df))
print("Filas ventas_powerbi:", len(ventas_powerbi))

print("Ventas netas en df:", round(df["venta"].sum(), 2))
print("Ventas netas en Power BI:", round(ventas_powerbi["venta"].sum(), 2))

diferencia = df["venta"].sum() - ventas_powerbi["venta"].sum()
print("Diferencia:", round(diferencia, 2))

## Conclusiones

El análisis con Python permitió caracterizar el comportamiento de ventas, clientes, productos, métodos de pago, categorías y rentabilidad a partir de los datos depurados en PostgreSQL.

Los resultados generados en este notebook fueron utilizados como base para la preparación de datos y visualizaciones desarrolladas posteriormente en Power BI.